In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [10]:
WORKING_DIRECTORY = "/Users/khatereh/Desktop/Research/ProcessModeling/PEEK/Large_System/SuperCellPEEK/ShearM/1"
CSV_FILENAME = "PEEKCrSyz_1_Stress_Strain_Data.csv"
ENDPOINT = 1000

In [11]:
os.chdir(WORKING_DIRECTORY)

In [12]:
sh_filename = CSV_FILENAME.split(".")[0]
sh_dat = pd.read_csv(CSV_FILENAME)

In [13]:
etruexy = sh_dat.iloc[:, 1].values
etruexz = sh_dat.iloc[:, 2].values
etrueyz = sh_dat.iloc[:, 3].values
sxy = sh_dat.iloc[:, 7].values
sxz = sh_dat.iloc[:, 8].values
syz = sh_dat.iloc[:, 9].values
direction = sh_dat.iloc[:, 10].values

print("\nDetermining shear direction...\n")



Determining shear direction...



In [14]:
if direction[0] == 1:
    primary_strain = etruexy
    primary_stress = sxy
    G_label = "G_xy"
    G_name = f"{sh_filename}_sxy_etruexy"

elif direction[0] == 2:
    primary_strain = etruexz
    primary_stress = sxz
    G_label = "G_xz"
    G_name = f"{sh_filename}_sxz_etruexz"

elif direction[0] == 3:
    primary_strain = etrueyz
    primary_stress = syz
    G_label = "G_yz"
    G_name = f"{sh_filename}_syz_etrueyz"

else:
    raise ValueError("Invalid direction value in CSV.")


In [15]:
strain = primary_strain[:ENDPOINT]
stress = primary_stress[:ENDPOINT]

In [16]:
def linear_fit(x, y):
    A = np.vstack([x, np.ones(len(x))]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    return slope, intercept

In [17]:
def find_best_breakpoint(x, y, min_index=50):
    best_error = np.inf
    best_index = None

    for i in range(min_index, len(x) - min_index):
        slope1, intercept1 = linear_fit(x[:i], y[:i])
        slope2, intercept2 = linear_fit(x[i:], y[i:])

        y1_pred = slope1 * x[:i] + intercept1
        y2_pred = slope2 * x[i:] + intercept2

        error = np.sum((y[:i] - y1_pred)**2) + np.sum((y[i:] - y2_pred)**2)

        if error < best_error:
            best_error = error
            best_index = i

    return best_index

In [18]:
break_index = find_best_breakpoint(strain, stress)

# Fit first (elastic) region
G_value, G_intercept = linear_fit(strain[:break_index], stress[:break_index])

yield_strain_value = strain[break_index]
yield_stress_value = G_value * yield_strain_value + G_intercept

# Convert to GPa
G_display = round(G_value / 1000.0, 2)


In [19]:
results = pd.DataFrame({
    "Shear_Modulus_MPa": [G_value],
    "Yield_Strain": [yield_strain_value],
    "Yield_Stress_MPa": [yield_stress_value],
    "Intercept": [G_intercept]
})

results.to_csv(f"{G_name}_Numeric_Values.txt", sep="\t", index=False)


In [20]:
plt.figure(figsize=(6,5))
plt.scatter(primary_strain, primary_stress, s=5, color="black")

x_fit = np.linspace(0, max(primary_strain), 200)
plt.plot(x_fit, G_value * x_fit + G_intercept, color="red", linewidth=2)

plt.scatter(yield_strain_value, yield_stress_value, s=80)

plt.title(f"{G_label} = {G_display} GPa")
plt.xlabel("True Shear Strain")
plt.ylabel("True Shear Stress (MPa)")

plt.savefig(f"{G_name}_Plot.pdf")
plt.close()

print("\nShear modulus analysis complete!")
print(f"Shear modulus: {G_display} GPa")
print(f"Plot saved: {G_name}_Plot.pdf")
print(f"Results saved: {G_name}_Numeric_Values.txt")



Shear modulus analysis complete!
Shear modulus: 0.77 GPa
Plot saved: PEEKCrSyz_1_Stress_Strain_Data_syz_etrueyz_Plot.pdf
Results saved: PEEKCrSyz_1_Stress_Strain_Data_syz_etrueyz_Numeric_Values.txt
